# Sesión 03 – ETL Escalable con Spark aplicado al Proyecto de Ventas

**Objetivo:** Construir un flujo **ETL completo** (Extract → Transform → Load) usando Apache Spark sobre el dataset real de ventas de facturas electrónicas 2021-2026, aplicando transformaciones encadenadas, joins distribuidos, funciones ventana, validación de calidad y guardado en formato **Parquet**.

**Dataset:** `Ventas_2021-2026.xlsx` — archivo Excel con transacciones de facturas electrónicas del negocio.

---

## ¿Qué es el ETL?

**ETL** son las siglas de **Extract, Transform, Load** (Extraer, Transformar, Cargar). Es el proceso estándar en Big Data para mover y preparar datos desde su fuente original hasta un destino analítico.

```
┌─────────────────────────────────────────────────────────────┐
│                    FLUJO ETL COMPLETO                        │
│                                                              │
│  ┌──────────────┐   ┌──────────────┐   ┌───────────────┐   │
│  │   EXTRACT    │   │  TRANSFORM   │   │     LOAD      │   │
│  │              │──▶│              │──▶│               │   │
│  │ Leer Excel   │   │ Limpiar      │   │ Guardar en    │   │
│  │ → pandas     │   │ Filtrar      │   │ Parquet       │   │
│  │ → Spark DF   │   │ Enriquecer   │   │               │   │
│  └──────────────┘   │ Calcular     │   └───────────────┘   │
│                     └──────────────┘                        │
└─────────────────────────────────────────────────────────────┘
```

| Fase | ¿Qué hace? | En nuestro proyecto |
|---|---|---|
| **Extract** | Lee los datos desde la fuente original | Cargamos el `.xlsx` con pandas y lo pasamos a Spark |
| **Transform** | Limpia, filtra y enriquece los datos | Eliminamos nulos, anulados, convertimos fechas, creamos columnas calculadas |
| **Load** | Guarda el resultado en el destino final | Escribimos el DataFrame procesado en formato Parquet |

**Temas de esta sesión:**
- Extract: Excel → pandas → Spark DataFrame
- Transform: limpieza, joins distribuidos, funciones ventana
- Validación de calidad de datos
- Load: escritura en Parquet
- ¿Por qué Parquet es mejor que CSV?
- Verificación de la salida

## Paso 1 — Inicializar SparkSession

La `SparkSession` es el punto de entrada al motor de Spark. La configuramos en modo local usando todos los núcleos disponibles (`local[*]`).

> **Recordatorio:** Spark usa **Lazy Evaluation** — las transformaciones no se ejecutan hasta que se invoca una **acción** (`.show()`, `.count()`, `.write`).

In [1]:
import pandas as pd
import warnings
import unicodedata
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_date, broadcast, when
from pyspark.sql.functions import sum as _sum, row_number, count
from pyspark.sql.window import Window

# Iniciamos SparkSession
spark = (
    SparkSession.builder
    .appName("ProyectoVentas_ETL")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"✅ SparkSession lista")
print(f"   Versión  : {spark.version}")
print(f"   App Name : {spark.sparkContext.appName}")
print(f"   Master   : {spark.sparkContext.master}")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 13:29:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/14 13:29:15 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/14 13:29:15 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


✅ SparkSession lista
   Versión  : 4.1.1
   App Name : ProyectoVentas_ETL
   Master   : local[*]


## Paso 2 — EXTRACT: Leer el dataset

### ¿Por qué usamos pandas primero?

Spark **no soporta archivos Excel (`.xlsx`) de forma nativa**. Por eso seguimos este pipeline de ingesta:

```
Ventas_2021-2026.xlsx
        │
        ▼ pandas.read_excel()   ← lee el Excel
  pandas DataFrame
        │
        ▼ spark.createDataFrame()  ← convierte a Spark
  Spark DataFrame   ✅ listo para ETL distribuido
```

También normalizamos los nombres de columnas (quitamos tildes, espacios → guion bajo) para evitar problemas en Spark.

In [2]:
EXCEL_PATH = "../data/Ventas_2021-2026.xlsx"

# ── Función para normalizar nombres de columnas ──
def normalizar_columna(nombre):
    """Convierte nombres de columnas a formato snake_case sin tildes."""
    nombre = str(nombre).strip().lower()
    nombre = unicodedata.normalize('NFD', nombre)
    nombre = ''.join(c for c in nombre if unicodedata.category(c) != 'Mn')
    nombre = nombre.replace(' ', '_').replace('/', '_').replace('-', '_')
    return nombre

# ── EXTRACT: Leer Excel con pandas ──
print("⏳ [EXTRACT] Cargando Ventas_2021-2026.xlsx con pandas...")
df_pd = pd.read_excel(EXCEL_PATH, dtype=str)
print(f"✅ Excel cargado → {df_pd.shape[0]:,} filas × {df_pd.shape[1]} columnas")

# Normalizar columnas
df_pd.columns = [normalizar_columna(c) for c in df_pd.columns]
print(f"\n📋 Columnas normalizadas:")
for i, c in enumerate(df_pd.columns, 1):
    print(f"   {i:>2}. {c}")

⏳ [EXTRACT] Cargando Ventas_2021-2026.xlsx con pandas...
✅ Excel cargado → 525,434 filas × 24 columnas

📋 Columnas normalizadas:
    1. fecha_de_emision
    2. tipo
    3. serie
    4. numero
    5. doc_entidad_tipo
    6. doc_entidad_numero
    7. denominacion_entidad
    8. moneda
    9. tipo_de_cambio
   10. unidad_de_medida
   11. codigo
   12. descripcion
   13. cantidad
   14. valor_unitario
   15. precio_unitario
   16. descuento
   17. subtotal
   18. tipo_de_igv
   19. igv
   20. tipo_de_isc
   21. isc
   22. impuesto_bolsas
   23. total
   24. anulado


In [10]:
# ── Convertir de pandas → Spark DataFrame ──
print("⏳ Convirtiendo pandas DataFrame → Spark DataFrame...")
ventas = spark.createDataFrame(df_pd)

print(f"✅ Spark DataFrame creado")
print(f"   Particiones iniciales : {ventas.rdd.getNumPartitions()}")
print(f"   Total registros       : {ventas.count():,}")
print()
print("📋 Schema (estructura de columnas):")
ventas.printSchema()

⏳ Convirtiendo pandas DataFrame → Spark DataFrame...
✅ Spark DataFrame creado
   Particiones iniciales : 28


26/04/14 13:34:48 WARN TaskSetManager: Stage 20 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.


   Total registros       : 525,434

📋 Schema (estructura de columnas):
root
 |-- fecha_de_emision: string (nullable = true)
 |-- tipo: string (nullable = true)
 |-- serie: string (nullable = true)
 |-- numero: string (nullable = true)
 |-- doc_entidad_tipo: string (nullable = true)
 |-- doc_entidad_numero: string (nullable = true)
 |-- denominacion_entidad: string (nullable = true)
 |-- moneda: string (nullable = true)
 |-- tipo_de_cambio: string (nullable = true)
 |-- unidad_de_medida: string (nullable = true)
 |-- codigo: string (nullable = true)
 |-- descripcion: string (nullable = true)
 |-- cantidad: string (nullable = true)
 |-- valor_unitario: string (nullable = true)
 |-- precio_unitario: string (nullable = true)
 |-- descuento: string (nullable = true)
 |-- subtotal: string (nullable = true)
 |-- tipo_de_igv: string (nullable = true)
 |-- igv: string (nullable = true)
 |-- tipo_de_isc: double (nullable = true)
 |-- isc: string (nullable = true)
 |-- impuesto_bolsas: string (nu

In [11]:
# ── Detectar columnas clave automáticamente ──
col_fecha   = [c for c in ventas.columns if 'fecha_de_emision' in c]
col_total   = [c for c in ventas.columns if 'total' in c]
col_anulado = [c for c in ventas.columns if 'anulado' in c]
col_cliente = [c for c in ventas.columns if 'denominacion' in c or 'denominacion_entidad' in c]
col_doc     = [c for c in ventas.columns if 'doc_entidad_numero' in c and 'entidad' in c]
col_serie   = [c for c in ventas.columns if 'serie' in c]
col_desc    = [c for c in ventas.columns if 'descripcion' in c]
col_codigo  = [c for c in ventas.columns if 'codigo' in c]

# Referencias globales
FECHA_COL   = col_fecha[0]   if col_fecha   else None
TOTAL_COL   = col_total[-1]  if col_total   else None   # última = TOTAL general
ANULADO_COL = col_anulado[0] if col_anulado else None
CLIENTE_COL = col_cliente[0] if col_cliente else None
DOC_COL     = col_doc[0]     if col_doc     else None
SERIE_COL   = col_serie[0]   if col_serie   else None
DESC_COL    = col_desc[0]    if col_desc    else None

print("🔍 Columnas clave detectadas automáticamente:")
print(f"   Fecha       : {FECHA_COL}")
print(f"   Total       : {TOTAL_COL}")
print(f"   Anulado     : {ANULADO_COL}")
print(f"   Cliente     : {CLIENTE_COL}")
print(f"   Doc Número  : {DOC_COL}")
print(f"   Serie       : {SERIE_COL}")
print(f"   Descripción : {DESC_COL}")

# Mostrar primeras filas
print("\n📄 Primeras 5 filas del dataset:")
ventas.show(5, truncate=True)

🔍 Columnas clave detectadas automáticamente:
   Fecha       : fecha_de_emision
   Total       : total
   Anulado     : anulado
   Cliente     : denominacion_entidad
   Doc Número  : doc_entidad_numero
   Serie       : serie
   Descripción : descripcion

📄 Primeras 5 filas del dataset:
+----------------+----+-----+------+----------------+------------------+--------------------+------+--------------+----------------+-------------------+--------------------+--------+--------------+---------------+---------+-------------+-----------+-------------+-----------+---+---------------+--------+-------+
|fecha_de_emision|tipo|serie|numero|doc_entidad_tipo|doc_entidad_numero|denominacion_entidad|moneda|tipo_de_cambio|unidad_de_medida|             codigo|         descripcion|cantidad|valor_unitario|precio_unitario|descuento|     subtotal|tipo_de_igv|          igv|tipo_de_isc|isc|impuesto_bolsas|   total|anulado|
+----------------+----+-----+------+----------------+------------------+----------------

26/04/14 13:35:57 WARN TaskSetManager: Stage 23 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/usr/local/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe


## Paso 3 — Explorar estructura

Antes de transformar, verificamos el esquema real: tipos de datos actuales y una vista previa de los registros.

> En el ETL, esta exploración es crítica porque nos dice **qué necesita ser corregido** antes de proceder.

In [12]:
# Estadísticas descriptivas de las columnas numéricas clave
print("📊 Estadísticas del dataset original:")
print(f"   Total de registros : {ventas.count():,}")
print(f"   Total de columnas  : {len(ventas.columns)}")

# Contar registros anulados vs válidos
if ANULADO_COL:
    print(f"\n⚠️  Distribución por estado (columna '{ANULADO_COL}'):")
    ventas.groupBy(ANULADO_COL).count().orderBy("count", ascending=False).show()

# Contar nulos por columna clave
cols_revisar = [c for c in [FECHA_COL, TOTAL_COL, ANULADO_COL, DOC_COL] if c]
print(f"\n🔍 Nulos por columna clave (antes de limpiar):")
ventas.select([
    F.count(F.when(F.col(c).isNull() | (F.col(c) == 'None') | (F.col(c) == ''), c)).alias(f"nulos_{c}")
    for c in cols_revisar
]).show(truncate=False)

📊 Estadísticas del dataset original:


26/04/14 13:36:01 WARN TaskSetManager: Stage 24 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.


   Total de registros : 525,434
   Total de columnas  : 24

⚠️  Distribución por estado (columna 'anulado'):


26/04/14 13:36:02 WARN TaskSetManager: Stage 27 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.


+-------+------+
|anulado| count|
+-------+------+
|     NO|522688|
|     SI|  2746|
+-------+------+


🔍 Nulos por columna clave (antes de limpiar):


26/04/14 13:36:02 WARN TaskSetManager: Stage 30 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.


+----------------------+-----------+-------------+------------------------+
|nulos_fecha_de_emision|nulos_total|nulos_anulado|nulos_doc_entidad_numero|
+----------------------+-----------+-------------+------------------------+
|0                     |0          |0            |0                       |
+----------------------+-----------+-------------+------------------------+



## Paso 4 — TRANSFORM: Limpieza básica

### ¿Por qué es importante limpiar antes?

Los datos crudos tienen problemas típicos que, si no se corrigen, **contaminan los resultados**:

| Problema | ¿Qué hacemos? |
|---|---|
| Registros anulados (facturas canceladas) | Filtramos `ANULADO = 'NO'` |
| Totales cero o negativos | Filtramos `TOTAL > 0` |
| Nulos en columnas críticas | Eliminamos con `dropna()` |
| Fecha como texto | Convertimos con `to_date()` |
| Total como texto | Convertimos con `.cast("double")` |

> **Nota:** Cada `.withColumn()`, `.filter()`, `.dropna()` es una **TRANSFORMACIÓN** — Spark solo construye el plan, no ejecuta.

In [13]:
# ── TRANSFORM: Limpieza de datos ──
ventas_limpio = ventas

# 1. Convertir TOTAL a numérico
if TOTAL_COL:
    ventas_limpio = ventas_limpio.withColumn(
        TOTAL_COL,
        F.regexp_replace(F.col(TOTAL_COL), ',', '.').cast('double')
    )
    print(f"✅ TRANSFORMACIÓN: '{TOTAL_COL}' convertido a double")

# 2. Convertir FECHA a DateType
if FECHA_COL:
    ventas_limpio = ventas_limpio.withColumn(
        "FECHA_DT",
        F.to_date(F.col(FECHA_COL), "dd/MM/yyyy")
    )
    print(f"✅ TRANSFORMACIÓN: 'FECHA_DT' creada desde '{FECHA_COL}' (DateType)")

# 3. Eliminar nulos en columnas críticas
cols_no_null = [c for c in [DOC_COL, FECHA_COL, TOTAL_COL] if c]
ventas_limpio = ventas_limpio.dropna(subset=cols_no_null)
print(f"✅ TRANSFORMACIÓN: Nulos eliminados en columnas críticas")

# 4. Filtrar anulados y totales inválidos
if ANULADO_COL:
    ventas_limpio = ventas_limpio.filter(
        (~F.upper(F.col(ANULADO_COL)).isin("SI", "S", "1", "TRUE", "ANULADO")) &
        (F.col(TOTAL_COL) > 0)
    )
    print(f"✅ TRANSFORMACIÓN: Anulados y totales <= 0 filtrados")

# 5. Crear columnas temporales para análisis
ventas_limpio = (
    ventas_limpio
    .withColumn("anio",     F.year(F.col("FECHA_DT")))
    .withColumn("mes",      F.month(F.col("FECHA_DT")))
    .withColumn("anio_mes", F.date_format(F.col("FECHA_DT"), "yyyy-MM"))
)
print(f"✅ TRANSFORMACIÓN: Columnas temporales (anio, mes, anio_mes) creadas")

print()
print("⚠️  Todas las operaciones anteriores son TRANSFORMACIONES (lazy)")
print("   Spark aún NO ha ejecutado el plan — solo lo construyó")

✅ TRANSFORMACIÓN: 'total' convertido a double
✅ TRANSFORMACIÓN: 'FECHA_DT' creada desde 'fecha_de_emision' (DateType)
✅ TRANSFORMACIÓN: Nulos eliminados en columnas críticas
✅ TRANSFORMACIÓN: Anulados y totales <= 0 filtrados
✅ TRANSFORMACIÓN: Columnas temporales (anio, mes, anio_mes) creadas

⚠️  Todas las operaciones anteriores son TRANSFORMACIONES (lazy)
   Spark aún NO ha ejecutado el plan — solo lo construyó


In [14]:
# ── ACCIÓN: Verificar resultado de la limpieza ──
n_total = ventas.count()
n_limpios = ventas_limpio.count()

print("▶️  Ejecutando acción .count() — aquí Spark corre el plan completo:")
print(f"   Registros originales  : {n_total:,}")
print(f"   Registros limpios     : {n_limpios:,}")
print(f"   Registros eliminados  : {n_total - n_limpios:,}")

cols_mostrar = [c for c in ["FECHA_DT", DOC_COL, CLIENTE_COL, TOTAL_COL] if c]
print(f"\n📄 Muestra de datos limpios:")
ventas_limpio.select(*cols_mostrar).show(5)

26/04/14 13:36:20 WARN TaskSetManager: Stage 33 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
26/04/14 13:36:20 WARN TaskSetManager: Stage 36 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
[Stage 36:>                                                       (0 + 28) / 28]

▶️  Ejecutando acción .count() — aquí Spark corre el plan completo:
   Registros originales  : 525,434
   Registros limpios     : 522,688
   Registros eliminados  : 2,746

📄 Muestra de datos limpios:
+----------+------------------+--------------------+--------+
|  FECHA_DT|doc_entidad_numero|denominacion_entidad|   total|
+----------+------------------+--------------------+--------+
|2021-01-25|                62|MARGARITA PINEDA ...|13.00005|
|2021-01-25|          00000001| MIRIAM PINTO VARGAS|    92.0|
|2021-01-25|          00000002|YOSELIN JAIMEY PA...|    26.0|
|2021-01-25|          40358816|COILA PACOMPIA HILDA|13.00005|
|2021-01-25|          40358816|COILA PACOMPIA HILDA| 0.86667|
+----------+------------------+--------------------+--------+
only showing top 5 rows


26/04/14 13:36:21 WARN TaskSetManager: Stage 39 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/usr/local/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe


## Paso 5 — TRANSFORM: Join distribuido (tabla de clientes)

### ¿Qué es un Join distribuido?

En nuestro dataset, la información del cliente (número de documento y nombre) está en el mismo archivo de ventas. Extraemos una **tabla de clientes única** y la unimos de vuelta con las ventas.

### Optimización: Broadcast Join

Cuando una de las dos tablas es **pequeña** (como la lista de clientes únicos), Spark puede enviar esa tabla completa a cada nodo worker en lugar de hacer un **shuffle** (redistribución costosa de datos):

```
Sin broadcast (shuffle):           Con broadcast:
  Ventas → redistribuye           Clientes → copia a todos los nodos
  datos por cliente_id            Ventas  → procesa localmente
  ❌ Costoso en red               ✅ Mucho más eficiente
```

In [15]:
# Construimos la tabla de clientes únicos (dimensión)
if DOC_COL and CLIENTE_COL:
    clientes_limpio = (
        ventas_limpio
        .select(
            F.col(DOC_COL).alias("cliente_id"),
            F.col(CLIENTE_COL).alias("nombre_cliente")
        )
        .dropDuplicates(["cliente_id"])
    )

    n_clientes = clientes_limpio.count()
    print(f"✅ Tabla de clientes únicos creada: {n_clientes:,} clientes")
    print(f"   Esta tabla es pequeña → ideal para Broadcast Join")
    clientes_limpio.show(5)
else:
    print("⚠️  No se detectaron columnas de cliente/documento")
    clientes_limpio = None

26/04/14 13:36:28 WARN TaskSetManager: Stage 40 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
26/04/14 13:36:29 WARN TaskSetManager: Stage 46 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.


✅ Tabla de clientes únicos creada: 16,869 clientes
   Esta tabla es pequeña → ideal para Broadcast Join


[Stage 46:>                                                       (0 + 28) / 28]

+----------+--------------------+
|cliente_id|      nombre_cliente|
+----------+--------------------+
|         .|       AGRIPINA YAPO|
|  00000000|BODEGUITA 3 DE MA...|
|  00000001| MIRIAM PINTO VARGAS|
|  00000002|YOSELIN JAIMEY PA...|
|  00000005|    GOYA VILMA VILCA|
+----------+--------------------+
only showing top 5 rows


In [16]:
# ── Join distribuido con Broadcast ──
if clientes_limpio is not None and DOC_COL:
    df_join = ventas_limpio.join(
        broadcast(clientes_limpio),           # ← tabla pequeña se envía a todos los nodos
        ventas_limpio[DOC_COL] == clientes_limpio["cliente_id"],
        "inner"                               # solo registros que coinciden
    )

    cols_join = [c for c in ["cliente_id", "nombre_cliente", "FECHA_DT", TOTAL_COL] if c]
    print("✅ Join completado (Broadcast Join):")
    df_join.select(*cols_join).show(5)
    print(f"   Total registros tras el join: {df_join.count():,}")
else:
    df_join = ventas_limpio
    print("⚠️  Join omitido — se continúa con ventas_limpio")

✅ Join completado (Broadcast Join):


26/04/14 13:36:34 WARN TaskSetManager: Stage 49 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
26/04/14 13:36:35 WARN TaskSetManager: Stage 52 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/usr/local/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe
26/04/14 13:36:35 WARN TaskSetManager: Stage 53 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.


+----------+--------------------+----------+--------+
|cliente_id|      nombre_cliente|  FECHA_DT|   total|
+----------+--------------------+----------+--------+
|        62|MARGARITA PINEDA ...|2021-01-25|13.00005|
|  00000001| MIRIAM PINTO VARGAS|2021-01-25|    92.0|
|  00000002|YOSELIN JAIMEY PA...|2021-01-25|    26.0|
|  40358816|COILA PACOMPIA HILDA|2021-01-25|13.00005|
|  40358816|COILA PACOMPIA HILDA|2021-01-25| 0.86667|
+----------+--------------------+----------+--------+
only showing top 5 rows


26/04/14 13:36:36 WARN TaskSetManager: Stage 56 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.


   Total registros tras el join: 522,688


## Paso 6 — TRANSFORM: Función Ventana

### ¿Qué es una función ventana?

A diferencia de `groupBy` (que **colapsa** las filas en una sola por grupo), una **función ventana** calcula métricas sobre cada fila **sin perder ninguna fila**.

| Operación | Resultado |
|---|---|
| `groupBy().agg()` | Una fila por grupo — pierde el detalle |
| Window function | Todas las filas — agrega la métrica como columna nueva |

Calculamos:
- `ranking`: orden de compra de cada cliente (1ª compra, 2ª compra, ...)
- `total_acumulado`: monto acumulado por cliente hasta cada fecha

In [17]:
# ── Definir la ventana: partición por cliente, ordenada por fecha ──
nombre_id_col = "cliente_id" if "cliente_id" in df_join.columns else DOC_COL

if nombre_id_col and TOTAL_COL:
    window_spec = Window.partitionBy(nombre_id_col).orderBy("FECHA_DT")

    df_window = (
        df_join
        # ranking: número de compra de ese cliente
        .withColumn("ranking", row_number().over(window_spec))
        # total acumulado: suma corriente de ventas del cliente
        .withColumn("total_acumulado", _sum(TOTAL_COL).over(window_spec))
    )

    cols_ventana = [c for c in ["FECHA_DT", "nombre_cliente", TOTAL_COL, "ranking", "total_acumulado"] if c in df_window.columns]
    print("📊 Métricas calculadas con Funciones Ventana:")
    print("   - ranking          : orden cronológico de compras del cliente")
    print("   - total_acumulado  : suma corriente de ventas por cliente")
    print()
    df_window.select(*cols_ventana).show(10)
else:
    df_window = df_join
    print("⚠️  Función ventana omitida — columnas clave no encontradas")

📊 Métricas calculadas con Funciones Ventana:
   - ranking          : orden cronológico de compras del cliente
   - total_acumulado  : suma corriente de ventas por cliente



26/04/14 13:36:42 WARN TaskSetManager: Stage 59 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
26/04/14 13:36:42 WARN TaskSetManager: Stage 62 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
[Stage 62:==>                                                     (1 + 27) / 28]

+----------+----------------+-------+-------+---------------+
|  FECHA_DT|  nombre_cliente|  total|ranking|total_acumulado|
+----------+----------------+-------+-------+---------------+
|2021-03-22|   AGRIPINA YAPO|   13.0|      1|           13.0|
|2021-04-15|   AGRIPINA YAPO|6.06669|      2|       26.00005|
|2021-04-15|   AGRIPINA YAPO|6.93336|      3|       26.00005|
|2022-03-04|   AGRIPINA YAPO|   13.0|      4|       62.86672|
|2022-03-04|   AGRIPINA YAPO|0.86667|      5|       62.86672|
|2022-03-04|   AGRIPINA YAPO|   23.0|      6|       62.86672|
|2021-01-25|GOYA VILMA VILCA|   21.0|      1|           21.0|
|2021-03-08|GOYA VILMA VILCA|   23.0|      2|       73.83333|
|2021-03-08|GOYA VILMA VILCA|   26.0|      3|       73.83333|
|2021-03-08|GOYA VILMA VILCA|3.83333|      4|       73.83333|
+----------+----------------+-------+-------+---------------+
only showing top 10 rows


## Paso 7 — Validación de Calidad de Datos

Antes de guardar, validamos que el resultado sea confiable:
1. **Nulos por columna** — ninguna columna crítica debe tener nulos
2. **Total de registros** — cuántos registros finales procesamos
3. **Rango de fechas** — cubrimos el período esperado (2021-2026)

In [18]:
# ── Validación 1: Nulos por columna clave ──
cols_validar = [c for c in ["FECHA_DT", TOTAL_COL, "total_acumulado"] if c in df_window.columns]

print("🔍 Validación 1 — Nulos por columna clave (deben ser 0):")
df_window.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in cols_validar
]).show()

# ── Validación 2: Total de registros ──
total_final = df_window.count()
print(f"🔍 Validación 2 — Total de registros finales procesados: {total_final:,}")

# ── Validación 3: Rango de fechas ──
if "FECHA_DT" in df_window.columns:
    rango = df_window.agg(
        F.min("FECHA_DT").alias("fecha_min"),
        F.max("FECHA_DT").alias("fecha_max")
    ).collect()[0]
    print(f"🔍 Validación 3 — Rango de fechas: {rango['fecha_min']} → {rango['fecha_max']}")

🔍 Validación 1 — Nulos por columna clave (deben ser 0):


26/04/14 13:36:56 WARN TaskSetManager: Stage 65 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
26/04/14 13:36:57 WARN TaskSetManager: Stage 68 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
26/04/14 13:36:58 WARN TaskSetManager: Stage 74 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.


+--------+-----+---------------+
|FECHA_DT|total|total_acumulado|
+--------+-----+---------------+
|       0|    0|              0|
+--------+-----+---------------+



26/04/14 13:36:59 WARN TaskSetManager: Stage 77 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
26/04/14 13:37:00 WARN TaskSetManager: Stage 80 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.


🔍 Validación 2 — Total de registros finales procesados: 522,688


26/04/14 13:37:01 WARN TaskSetManager: Stage 83 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
[Stage 83:>                                                       (0 + 28) / 28]

🔍 Validación 3 — Rango de fechas: 2021-01-25 → 2026-04-06


In [19]:
# ── Aggregaciones batch: resumen de ventas por año ──
if TOTAL_COL and "anio" in df_window.columns:
    print("📊 Ventas por año (agregación batch final):")
    df_window.groupBy("anio").agg(
        F.count("*").alias("num_facturas"),
        F.sum(TOTAL_COL).alias("total_ventas_soles"),
        F.avg(TOTAL_COL).alias("promedio_venta")
    ).filter(F.col("anio").isNotNull()).orderBy("anio").show(truncate=False)

📊 Ventas por año (agregación batch final):


26/04/14 13:37:03 WARN TaskSetManager: Stage 86 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
26/04/14 13:37:04 WARN TaskSetManager: Stage 89 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
[Stage 89:>                                                       (0 + 28) / 28]

+----+------------+------------------+------------------+
|anio|num_facturas|total_ventas_soles|promedio_venta    |
+----+------------+------------------+------------------+
|2021|126411      |2322124.8681659936|18.369642421672115|
|2022|122011      |2263605.3255835464|18.552469249359046|
|2023|98207       |4759104.070443967 |48.459927199119896|
|2024|98740       |6028367.379077533 |61.05294084542772 |
|2025|72328       |1.81830974566502E7|251.3977637519384 |
|2026|4991        |2590502.020160006 |519.034666431578  |
+----+------------+------------------+------------------+



## Paso 8 — Revisar el Plan de Ejecución (Performance)

`explain()` nos muestra cómo Spark **optimiza internamente** nuestras operaciones. Es útil para entender:
- Qué tipo de join eligió Spark (BroadcastHashJoin = bueno)
- Si hay operaciones de shuffle (Exchange) costosas
- Cómo Catalyst Optimizer reorganizó el plan

In [20]:
# Ver el plan de ejecución físico
print("🔍 Plan de ejecución (Physical Plan):")
print("=" * 70)
# Solo el physical plan (False = resumen)
df_window.explain(False)

# Número de particiones
print(f"\n📦 Particiones del DataFrame final: {df_window.rdd.getNumPartitions()}")
print("   (cada partición = un bloque de datos procesado en paralelo)")

🔍 Plan de ejecución (Physical Plan):
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Window [row_number() windowspecdefinition(cliente_id#721, FECHA_DT#644 ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS ranking#840, sum(total#643) windowspecdefinition(cliente_id#721, FECHA_DT#644 ASC NULLS FIRST, specifiedwindowframe(RangeFrame, unboundedpreceding$(), currentrow$())) AS total_acumulado#842], [cliente_id#721], [FECHA_DT#644 ASC NULLS FIRST]
   +- Sort [cliente_id#721 ASC NULLS FIRST, FECHA_DT#644 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(cliente_id#721, 8), ENSURE_REQUIREMENTS, [plan_id=2314]
         +- BroadcastHashJoin [doc_entidad_numero#427], [cliente_id#721], Inner, BuildRight, false
            :- Project [fecha_de_emision#422, tipo#423, serie#424, numero#425, doc_entidad_tipo#426, doc_entidad_numero#427, denominacion_entidad#428, moneda#429, tipo_de_cambio#430, unidad_de_medida#431, codigo#432, descripcion

26/04/14 13:37:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/14 13:37:10 WARN TaskSetManager: Stage 92 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
26/04/14 13:37:11 WARN TaskSetManager: Stage 95 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
[Stage 95:>                                                       (0 + 28) / 28]


📦 Particiones del DataFrame final: 8
   (cada partición = un bloque de datos procesado en paralelo)


## Paso 9 — LOAD: Guardar en formato Parquet

### ¿Qué es Parquet y para qué sirve?

**Parquet** es un formato de almacenamiento **columnar** diseñado específicamente para Big Data y analítica.

```
CSV (formato fila):                    Parquet (formato columnar):
┌────┬──────┬──────┬───────┐           ┌────────────┐ ┌──────────┐ ┌───────┐
│ id │fecha │total │cliente│           │ id:1,2,3.. │ │fecha:... │ │total: │
├────┼──────┼──────┼───────┤           │            │ │          │ │...... │
│  1 │01/01 │23.0  │Juan   │           └────────────┘ └──────────┘ └───────┘
│  2 │01/01 │13.0  │María  │            Cada columna → bloque separado
└────┴──────┴──────┴───────┘
```

### ¿Por qué Parquet es mejor que CSV para analítica?

| Característica | CSV | Parquet |
|---|---|---|
| **Compresión** | Sin compresión nativa | 5x-10x más comprimido |
| **Velocidad de lectura** | Lee TODAS las columnas | Lee SOLO las columnas que necesitas |
| **Schema** | Sin tipos — todo es texto | Tipos de datos guardados |
| **Particionamiento** | No | Permite particionar por fecha, año, etc. |
| **Compatibilidad** | Universal | Spark, Hive, Presto, BigQuery, Pandas |
| **Uso ideal** | Intercambio de datos | Analítica batch, Data Lake, Data Warehouse |

> **En resumen:** Si CSV es como guardar ropa en una maleta sin doblar, Parquet es como doblar y organizar por tipo. Cuando buscas una camisa, en CSV revisas toda la maleta; en Parquet vas directo al compartimento de camisas.

In [21]:
import os

# ── LOAD: Guardar resultado en Parquet ──
# Definimos la ruta de salida dentro de artifacts del proyecto
output_path = "../artifacts/output_etl_ventas"

print(f"💾 [LOAD] Guardando datos procesados en formato Parquet...")
print(f"   Ruta de destino : {output_path}")
print(f"   Modo            : overwrite (sobreescribe si ya existe)")
print()

# Escribir en Parquet
# mode('overwrite') → si ya existe el directorio, lo reemplaza
df_window.write.mode("overwrite").parquet(output_path)

print(f"✅ ETL completado exitosamente.")
print(f"   Datos guardados en: {output_path}")
print()
print("📁 ¿Qué se guardó?")
print("   Spark crea una CARPETA (no un solo archivo) con:")
print("   ├── part-00000-xxxx.snappy.parquet  ← datos partición 0")
print("   ├── part-00001-xxxx.snappy.parquet  ← datos partición 1")
print("   ├── ...")
print("   └── _SUCCESS                        ← indica escritura exitosa")

💾 [LOAD] Guardando datos procesados en formato Parquet...
   Ruta de destino : ../artifacts/output_etl_ventas
   Modo            : overwrite (sobreescribe si ya existe)



26/04/14 13:37:20 WARN TaskSetManager: Stage 96 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
26/04/14 13:37:21 WARN TaskSetManager: Stage 99 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
26/04/14 13:37:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

✅ ETL completado exitosamente.
   Datos guardados en: ../artifacts/output_etl_ventas

📁 ¿Qué se guardó?
   Spark crea una CARPETA (no un solo archivo) con:
   ├── part-00000-xxxx.snappy.parquet  ← datos partición 0
   ├── part-00001-xxxx.snappy.parquet  ← datos partición 1
   ├── ...
   └── _SUCCESS                        ← indica escritura exitosa


## Paso 10 — Verificación de la salida Parquet

Leemos el Parquet recién guardado para comprobar que el ETL se ejecutó correctamente. Esta es la forma estándar de **validar** el paso Load.

In [22]:
# ── Leer el Parquet guardado y verificar ──
print("⏳ Leyendo Parquet desde disco...")
salida_final = spark.read.parquet(output_path)

print(f"✅ Parquet leído correctamente")
print(f"   Registros recuperados : {salida_final.count():,}")
print(f"   Columnas              : {len(salida_final.columns)}")
print()
print("📋 Schema del Parquet (los tipos se preservaron):")
salida_final.printSchema()

# Muestra de datos
cols_resultado = [c for c in ["cliente_id", "nombre_cliente", "FECHA_DT", TOTAL_COL, "total_acumulado"] if c in salida_final.columns]
print("\n📄 Muestra de datos recuperados desde Parquet:")
salida_final.select(*cols_resultado).show(10)

⏳ Leyendo Parquet desde disco...
✅ Parquet leído correctamente
   Registros recuperados : 522,688
   Columnas              : 32

📋 Schema del Parquet (los tipos se preservaron):
root
 |-- fecha_de_emision: string (nullable = true)
 |-- tipo: string (nullable = true)
 |-- serie: string (nullable = true)
 |-- numero: string (nullable = true)
 |-- doc_entidad_tipo: string (nullable = true)
 |-- doc_entidad_numero: string (nullable = true)
 |-- denominacion_entidad: string (nullable = true)
 |-- moneda: string (nullable = true)
 |-- tipo_de_cambio: string (nullable = true)
 |-- unidad_de_medida: string (nullable = true)
 |-- codigo: string (nullable = true)
 |-- descripcion: string (nullable = true)
 |-- cantidad: string (nullable = true)
 |-- valor_unitario: string (nullable = true)
 |-- precio_unitario: string (nullable = true)
 |-- descuento: string (nullable = true)
 |-- subtotal: string (nullable = true)
 |-- tipo_de_igv: string (nullable = true)
 |-- igv: string (nullable = true)
 |-

In [23]:
# ── Comparación CSV vs Parquet ──
import os

# Guardar también como CSV para comparar tamaños
csv_path = "../artifacts/output_etl_ventas_csv"
print("⏳ Guardando también en CSV para comparar tamaños...")
df_window.write.mode("overwrite").option("header", "true").csv(csv_path)

# Calcular tamaños
def calc_size_mb(path):
    total = 0
    if os.path.exists(path):
        for f in os.listdir(path):
            fp = os.path.join(path, f)
            if os.path.isfile(fp):
                total += os.path.getsize(fp)
    return total / (1024 * 1024)

size_parquet = calc_size_mb(output_path)
size_csv     = calc_size_mb(csv_path)

print()
print("📊 Comparación de tamaños (mismo dataset):")
print(f"   CSV     : {size_csv:.2f} MB")
print(f"   Parquet : {size_parquet:.2f} MB")
if size_parquet > 0:
    ratio = size_csv / size_parquet
    print(f"   Parquet es {ratio:.1f}x más pequeño que CSV ✅")
print()
print("💡 Parquet también es más rápido de leer porque Spark")
print("   solo carga las columnas que necesitas (lectura columnar).")

⏳ Guardando también en CSV para comparar tamaños...


26/04/14 13:37:46 WARN TaskSetManager: Stage 107 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
26/04/14 13:37:47 WARN TaskSetManager: Stage 110 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
                                                                                


📊 Comparación de tamaños (mismo dataset):
   CSV     : 122.73 MB
   Parquet : 10.81 MB
   Parquet es 11.4x más pequeño que CSV ✅

💡 Parquet también es más rápido de leer porque Spark
   solo carga las columnas que necesitas (lectura columnar).


## Paso 11 — Análisis adicional desde el Parquet

Una vez guardado el Parquet, podemos leerlo para análisis adicionales sin repetir todo el ETL. Aquí demostramos cómo agregar **categorías de valor por cliente**.

In [24]:
# ── Análisis sobre el Parquet guardado ──
if "total_acumulado" in salida_final.columns:
    df_reporte = (
        salida_final
        .withColumn(
            "categoria_cliente",
            when(col("total_acumulado") < 1000, "🟢 Básico")
            .when(col("total_acumulado") <= 5000, "🟡 Frecuente")
            .otherwise("🔴 Premium")
        )
    )

    print("🏆 Top 10 clientes por total acumulado con categoría:")
    cols_rep = [c for c in ["nombre_cliente", "total_acumulado", "categoria_cliente"] if c in df_reporte.columns]
    df_reporte.select(*cols_rep).orderBy(F.desc("total_acumulado")).limit(10).show(truncate=False)

    print("\n📊 Distribución de clientes por categoría:")
    df_reporte.groupBy("categoria_cliente").agg(
        F.countDistinct("cliente_id" if "cliente_id" in df_reporte.columns else DOC_COL).alias("num_clientes")
    ).orderBy("categoria_cliente").show()

🏆 Top 10 clientes por total acumulado con categoría:
+--------------------------------+------------------+-----------------+
|nombre_cliente                  |total_acumulado   |categoria_cliente|
+--------------------------------+------------------+-----------------+
|PETI DISTRIBUCIONES E.I.R.L.    |7776010.0         |🔴 Premium       |
|PETI DISTRIBUCIONES E.I.R.L.    |7088010.0         |🔴 Premium       |
|PETI DISTRIBUCIONES E.I.R.L.    |6400010.0         |🔴 Premium       |
|DISTRIBUCIONES OURS-HUA E.I.R.L.|3900008.0         |🔴 Premium       |
|DISTRIBUCIONES OURS-HUA E.I.R.L.|3900008.0         |🔴 Premium       |
|DISTRIBUCIONES OURS-HUA E.I.R.L.|3900008.0         |🔴 Premium       |
|CLIENTES VARIOS                 |3711315.0313398303|🔴 Premium       |
|CLIENTES VARIOS                 |3710899.0313398303|🔴 Premium       |
|CLIENTES VARIOS                 |3710899.0313398303|🔴 Premium       |
|CLIENTES VARIOS                 |3710899.0313398303|🔴 Premium       |
+--------------------

In [25]:
# ── Top 3 meses con más ventas (desde el Parquet) ──
if "anio_mes" in salida_final.columns and TOTAL_COL in salida_final.columns:
    print("🏆 Top 3 meses con más ventas (leído desde Parquet):")
    salida_final.groupBy("anio_mes").agg(
        F.sum(TOTAL_COL).alias("total_ventas"),
        F.count("*").alias("num_registros")
    ).filter(F.col("anio_mes").isNotNull()) \
     .orderBy(F.desc("total_ventas")) \
     .limit(3) \
     .show(truncate=False)

🏆 Top 3 meses con más ventas (leído desde Parquet):
+--------+-----------------+-------------+
|anio_mes|total_ventas     |num_registros|
+--------+-----------------+-------------+
|2025-11 |5890350.1454     |4594         |
|2025-10 |3884992.46459    |2957         |
|2025-12 |3322840.840079998|4441         |
+--------+-----------------+-------------+



## Paso 12 — Cierre: Resumen del ETL

### ¿Qué hicimos en esta sesión?

| Fase ETL | Paso | Operación | Detalles |
|---|---|---|---|
| **Extract** | 2 | Leer Excel | `pandas.read_excel()` → normalizar columnas → `spark.createDataFrame()` |
| **Extract** | 3 | Explorar estructura | `printSchema()`, detectar nulos, distribución de anulados |
| **Transform** | 4 | Limpieza | Convertir tipos, eliminar nulos, filtrar anulados y totales <= 0 |
| **Transform** | 5 | Join distribuido | Tabla de clientes + `broadcast()` para optimización |
| **Transform** | 6 | Función ventana | `ranking` y `total_acumulado` por cliente |
| **Transform** | 7 | Validación | Nulos = 0, total de registros, rango de fechas |
| **Transform** | 8 | Plan de ejecución | `explain()` para ver BroadcastHashJoin y Exchange |
| **Load** | 9 | Guardar Parquet | `df_window.write.mode('overwrite').parquet(path)` |
| **Load** | 10 | Verificar salida | Leer el Parquet y confirmar registros y schema |

### ¿Por qué Parquet es la salida ideal?

- ✅ **Compresión automática** — hasta 10x más pequeño que CSV
- ✅ **Lectura columnar** — Spark solo lee las columnas que necesitas
- ✅ **Schema preservado** — los tipos de datos se guardan con los datos
- ✅ **Compatible** con Spark, Pandas, BigQuery, Hive, Athena
- ✅ **Base para el Notebook 04** — HDFS y formatos de almacenamiento

### Preguntas para reflexión:
1. ¿En qué momento Spark ejecutó realmente el plan de transformaciones?
2. ¿Qué ventaja tuvo el Broadcast Join sobre un join normal?
3. ¿Qué diferencia hay entre `groupBy().agg()` y una función ventana?
4. ¿Por qué Parquet es mejor que CSV para el siguiente notebook de HDFS?

---

**Próximo Notebook (04):** HDFS y formatos de almacenamiento — Parquet con particionamiento, ORC, Delta Lake.

---
### Resumen del pipeline ejecutado:

| Paso | Tarea | Estado |
|---|---|---|
| 1 | SparkSession configurada | ✅ |
| 2 | Excel → pandas → Spark DataFrame | ✅ |
| 3 | Exploración y detección de columnas clave | ✅ |
| 4 | Limpieza: tipos, nulos, anulados, columnas temporales | ✅ |
| 5 | Join distribuido con Broadcast (tabla clientes) | ✅ |
| 6 | Funciones ventana: ranking y total acumulado | ✅ |
| 7 | Validación de calidad: nulos, conteos, fechas | ✅ |
| 8 | Plan de ejecución analizado con `explain()` | ✅ |
| 9 | Output guardado en Parquet | ✅ |
| 10 | Parquet verificado y leído correctamente | ✅ |
| 11 | Análisis: categorías de cliente, top meses | ✅ |